# 🎓 ProctorAI - Demo Thuật Toán Đo Góc Đầu (Head Pose Estimation)
> **Samsung Innovation Campus (SIC) Capstone Project** | Thành viên: Hiếu & Rin
>
> Notebook này được tối giản gồm **đúng 2 ô code**:
> * **Ô 1:** Khởi tạo mô hình & Hàm giải thuật PnP toàn cục (`SOLVEPNP_SQPNP`).
> * **Ô 2:** Khởi chạy Webcam thời gian thực, vẽ 3 trục tọa độ và hiển thị góc nhìn trực tiếp (Nhấn **'q'** để dừng).


In [ ]:
# =====================================================================
# Ô 1: THUẬT TOÁN ĐO GÓC ĐẦU 3D (MediaPipe Face Mesh + solvePnP SQPNP)
# =====================================================================
import cv2
import mediapipe as mp
import numpy as np

# 1. Khởi tạo MediaPipe Face Mesh (đếm tối đa 4 người trong phòng thi)
mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    max_num_faces=4,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Chỉ số 6 mốc giải phẫu cốt lõi trên khuôn mặt
# 1: Đỉnh mũi | 199: Đáy cằm | 33, 263: Khóe mắt ngoài | 61, 291: Khóe miệng ngoài
FACE_KEYPOINTS = [1, 199, 33, 263, 61, 291]

# 3. Mô hình nhân trắc học 3D chuẩn khuôn mặt người (đơn vị: mm)
MODEL_POINTS_3D = np.array([
    (0.0, 0.0, 0.0),             # 1: Đỉnh mũi (Gốc tọa độ)
    (0.0, 110.0, 65.0),          # 199: Đáy cằm
    (-65.0, -50.0, 60.0),        # 33: Khóe mắt trái
    (65.0, -50.0, 60.0),         # 263: Khóe mắt phải
    (-40.0, 50.0, 40.0),         # 61: Khóe miệng trái
    (40.0, 50.0, 40.0)           # 291: Khóe miệng phải
], dtype=np.float64)

# 4. Hàm giải thuật đo góc đầu từ 1 khung hình RGB
def estimate_head_pose(frame_rgb, cam_w=640, cam_h=480):
    results = mp_face_mesh.process(frame_rgb)
    
    # Kiểm tra số lượng người trong phòng thi
    if not results.multi_face_landmarks:
        return 0.0, 0.0, "Mat mat / Che camera", None, None, None
    
    num_faces = len(results.multi_face_landmarks)
    if num_faces > 1:
        return 0.0, 0.0, f"Phat hien {num_faces} nguoi (Co nguoi thu 2)", None, None, None
    
    # Trích xuất 6 mốc 2D pixel
    landmarks = results.multi_face_landmarks[0].landmark
    pts_2d = np.array([
        [landmarks[idx].x * cam_w, landmarks[idx].y * cam_h]
        for idx in FACE_KEYPOINTS
    ], dtype=np.float64)
    
    # Ma trận camera nội suy K (fx=fy=cam_w, cx=cam_w/2, cy=cam_h/2)
    cam_matrix = np.array([
        [cam_w, 0, cam_w / 2],
        [0, cam_w, cam_h / 2],
        [0, 0, 1]
    ], dtype=np.float64)
    dist_coeffs = np.zeros((4, 1), dtype=np.float64)
    
    # Giải thuật SQPNP: Khớp 3D sang 2D tìm vector quay rot_vec và tịnh tiến trans_vec
    success, rot_vec, trans_vec = cv2.solvePnP(
        MODEL_POINTS_3D, pts_2d, cam_matrix, dist_coeffs, flags=cv2.SOLVEPNP_SQPNP
    )
    if not success:
        return 0.0, 0.0, None, None, None, None
    
    # Chuyển đổi sang Ma trận quay R (3x3) và phân rã Euler (RQ Decomposition)
    rot_mat, _ = cv2.Rodrigues(rot_vec)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rot_mat)
    
    # RQDecomp3x3 trả về trực tiếp theo đơn vị độ (°)
    pitch = round(float(angles[0]), 1) # Cúi / Ngửa
    yaw = round(float(angles[1]), 1)   # Trái / Phải
    
    # Phân loại hành vi theo ngưỡng chuẩn công thái học
    violation = None
    if abs(yaw) > 35.0:
        violation = "Quay han ra sau"
    elif yaw > 22.0:
        violation = "Quay sang trai"
    elif yaw < -22.0:
        violation = "Quay sang phai"
    elif pitch > 20.0:
        violation = "Cui xuong gam ban"
    elif pitch < -20.0:
        violation = "Ngua mat len tran"
        
    return pitch, yaw, violation, pts_2d, rot_vec, trans_vec

print("✅ Ô 1: Thuật toán đo góc đầu SQPNP đã sẵn sàng!")


In [ ]:
# =====================================================================
# Ô 2: CHẠY CAMERA TRỰC TIẾP THỜI GIAN THỰC (Nhấn 'q' để dừng)
# =====================================================================
cap = cv2.VideoCapture(0)
cam_w, cam_h = 640, 480
cap.set(cv2.CAP_PROP_FRAME_WIDTH, cam_w)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cam_h)

cam_matrix = np.array([[cam_w, 0, cam_w / 2], [0, cam_w, cam_h / 2], [0, 0, 1]], dtype=np.float64)
dist_coeffs = np.zeros((4, 1), dtype=np.float64)

# 3 Trục tọa độ 3D từ mũi: X (Đỏ - ngang), Y (Xanh lá - dọc), Z (Xanh dương - hướng nhìn)
axis_3d = np.array([(80.0, 0, 0), (0, 80.0, 0), (0, 0, 80.0)], dtype=np.float64)

print("🚀 Đang mở Webcam... Hãy quay đầu thử nghiệm! Nhấn phím 'q' trên cửa sổ để tắt.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame = cv2.flip(frame, 1) # Lật gương tự nhiên
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Gọi hàm đo góc đầu từ Ô 1
    pitch, yaw, violation, pts_2d, rot_vec, trans_vec = estimate_head_pose(frame_rgb, cam_w, cam_h)
    
    # Vẽ 3 trục tọa độ và các điểm mốc nếu có mặt
    if pts_2d is not None and rot_vec is not None:
        # Chiếu 3 trục 3D sang 2D pixel
        axis_2d, _ = cv2.projectPoints(axis_3d, rot_vec, trans_vec, cam_matrix, dist_coeffs)
        nose_pt = (int(pts_2d[0][0]), int(pts_2d[0][1]))
        x_pt = (int(axis_2d[0][0][0]), int(axis_2d[0][0][1]))
        y_pt = (int(axis_2d[1][0][0]), int(axis_2d[1][0][1]))
        z_pt = (int(axis_2d[2][0][0]), int(axis_2d[2][0][1]))
        
        # Vẽ trục X (Đỏ), Y (Xanh lá), Z (Xanh dương - Hướng nhìn từ mũi)
        cv2.line(frame, nose_pt, x_pt, (0, 0, 255), 2)   # X: Đỏ
        cv2.line(frame, nose_pt, y_pt, (0, 255, 0), 2)   # Y: Xanh lá
        cv2.line(frame, nose_pt, z_pt, (255, 180, 0), 3) # Z: Hướng nhìn
        
        # Vẽ 6 mốc giải phẫu
        for pt in pts_2d:
            cv2.circle(frame, (int(pt[0]), int(pt[1])), 4, (0, 255, 255), -1)
            
    # Vẽ thanh HUD trạng thái
    color = (0, 0, 255) if (violation and ("sau" in violation.lower() or "nguoi" in violation.lower())) \
            else (0, 255, 255) if violation else (0, 255, 0)
    status_text = f"CHU Y: {violation.upper()}" if violation else "BINH THUONG - HOP LE"
    
    cv2.rectangle(frame, (10, 10), (630, 80), (20, 20, 20), -1)
    cv2.rectangle(frame, (10, 10), (630, 80), color, 2)
    cv2.putText(frame, f"TRANG THAI: {status_text}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    cv2.putText(frame, f"Pitch: {pitch:+.1f} deg | Yaw: {yaw:+.1f} deg", (20, 68), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    cv2.imshow("ProctorAI - Demo Head Pose (Nhan 'q' de thoat)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("👋 Đã đóng Camera thành công!")
